# Team

<a target="_blank" href="https://colab.research.google.com/github/glaucogoncalves/nio/blob/main/assignments/ilp-exec-2.ipynb"> <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Please list student names here

# Attention, please!

Solve the problems below. Show the problem model and the solution using ```pyomo``` and the GLPK or COIN-OR solvers.

To see how to write optimization models in colab, look for examples in the Linear Programming class colab.

Please send your Colab file (.ipynb) in SIGAA.

In [1]:
%%capture
# Instalar o Pyomo
!pip install pyomo --disable-pip-version-check

# Instalar o solver GLPK
!apt-get install -y glpk-utils

# Problem 1

FFB fresh fruit business mixes apples, peaches, and nectarines to make three different types of baskets for local market. Each basket contains approximately 5 kg fruits. The content of three types of baskets is specified as follows:

|Basket Type|Apple|Peach|Nectarine|
|--|--|--|--|
|1|At least 30%|At most 20%|-|
|2|-|At most 40%|At least 20%|
|3|At least 20%|-|At most 30%|

FFB purchases apple at a cost of $1.00/kg, peach $1.50/kg, and nectarine $1.80/kg, and sells type-1 basket at $2.25/kg, type-2 basket $3.00/kg, and type-3 basket $2.60/kg. The daily supply of fruit is limited to 60 kg of apples, 70 kg of peaches, and 50 kg of nectarines.  FFB is able to sell all the fruit baskets they prepare for a given day.

Formulate an ILP model to determine how the fruit be mixed in order to maximize the profit. Solve it.

In [2]:
from pyomo.environ import *

# Modelo
model = ConcreteModel()

# Variáveis de decisão (inteiras)
model.x1 = Var(within=NonNegativeIntegers)
model.x2 = Var(within=NonNegativeIntegers)
model.x3 = Var(within=NonNegativeIntegers)

# Dados
basket_weight = 5
price = {1: 11.25, 2: 15.00, 3: 13.00}
cost = {
    1: (0.3*1.0 + 0.2*1.5 + 0.5*1.8) * basket_weight,
    2: (0.4*1.5 + 0.2*1.8 + 0.4*1.0) * basket_weight,
    3: (0.2*1.0 + 0.5*1.5 + 0.3*1.8) * basket_weight
}

# Função objetivo: maximizar lucro
model.obj = Objective(
    expr = (price[1] - cost[1])*model.x1
         + (price[2] - cost[2])*model.x2
         + (price[3] - cost[3])*model.x3,
    sense = maximize
)

# Restrições de matéria-prima
model.apple = Constraint(expr = basket_weight*(0.3*model.x1 + 0.4*model.x2 + 0.2*model.x3) <= 60)
model.peach = Constraint(expr = basket_weight*(0.2*model.x1 + 0.4*model.x2 + 0.5*model.x3) <= 70)
model.nectarine = Constraint(expr = basket_weight*(0.5*model.x1 + 0.2*model.x2 + 0.3*model.x3) <= 50)

# Solver
solver = SolverFactory('glpk')  # ou 'cbc'
solver.solve(model, tee=False)

# Resultados
print("Optimal solution:")
print(f"x1 (basket type 1): {model.x1.value}")
print(f"x2 (basket type 2): {model.x2.value}")
print(f"x3 (basket type 3): {model.x3.value}")
print(f"Maximum profit: R${model.obj():.2f}")


Optimal solution:
x1 (basket type 1): 0.0
x2 (basket type 2): 27.0
x3 (basket type 3): 6.0
Maximum profit: R$254.70


# Problem 2

A military base needs to deliver essential aid to three disaster areas (D1, D2, and D3) using a mixed fleet of four available transport aircraft (A1, A2, A3, and A4). The primary objective is to minimize the cost of the overall mission, without exceeding the aircraft capacity and assuming that each aircraft can perform several flights.

There is an amount of 350 tons of cargo to be delivered. The airplane characteristics is provided below:

| Aircraft | Cargo Capacity (tons) | Cost per Flight (USD)
| ---|---|---|
|A1|18|5000|
|A2|50|10000|
|A3|24|7000|
|A4|70|15000|

Following is the disaster area demands:

|Disaster Area | Cargo Required (tons) |
|---|---|
|D1|50|
|D2|200|
|D3|100|

Formulate an ILP model and solve it.

In [3]:
from pyomo.environ import *

# -----------------------------
# Model
# -----------------------------
model = ConcreteModel()

# Sets
aircraft = ['A1', 'A2', 'A3', 'A4']
disaster = ['D1', 'D2', 'D3']

# Parameters
capacity = {'A1': 18, 'A2': 50, 'A3': 24, 'A4': 70}
cost = {'A1': 5000, 'A2': 10000, 'A3': 7000, 'A4': 15000}
demand = {'D1': 50, 'D2': 200, 'D3': 100}

# Decision variables: integer number of flights
model.x = Var(aircraft, disaster, within=NonNegativeIntegers)

# Objective: minimize total cost
model.obj = Objective(
    expr = sum(cost[i]*model.x[i,j] for i in aircraft for j in disaster),
    sense = minimize
)

# Demand constraints
def demand_rule(model, j):
    return sum(capacity[i]*model.x[i,j] for i in aircraft) >= demand[j]
model.demand = Constraint(disaster, rule=demand_rule)

# -----------------------------
# Solve
# -----------------------------
solver = SolverFactory('glpk')  # or 'cbc'
solver.solve(model, tee=False)

# -----------------------------
# Display results
# -----------------------------
print("Optimal number of flights per aircraft to each disaster area:\n")
for i in aircraft:
    for j in disaster:
        if model.x[i,j].value > 0:
            print(f"{i} -> {j}: {model.x[i,j].value:.0f} flights")

total_cost = value(model.obj)
print(f"\nMinimum total cost: ${total_cost:,.2f}")


Optimal number of flights per aircraft to each disaster area:

A2 -> D1: 1 flights
A2 -> D2: 4 flights
A2 -> D3: 2 flights

Minimum total cost: $70,000.00


# Problem 3

A large supermarket chain in the UK needs to build warehouses for a set of supermarkets it is opening in Northern England. The locations of the supermarkets have been identified, but the locations of the warehouses have yet to be determined.

Several good candidate locations for the warehouses have been identified, but decisions must be made regarding how many warehouses to open and at which candidate locations to build them. Moreover, each supermarket has an specific demand that must be met by the warehouses. If one warehouse is not sufficient to meet the demand, other warehouses can chosen to supply the supermarket.

Opening many warehouses would be advantageous as this would reduce the average distance a truck has to drive from the warehouse to the supermarket, and hence reduce the delivery cost. However, opening a warehouse has a fixed cost associated with it.

Our goal is **to find the optimal tradeoff between delivery costs and the costs of building new facilities**.

Below is the set of supermarkets. The coordinates and demand of each supermarket are provided in the following table.

| <i></i> | Coordinates | Demand |
| --- | --- | --- |
| Supermarket 1 | (0,1.5) | 16,000 |
| Supermarket 2 | (2.5,1.2) | 18,000 |
| Supermarket 3 | (4,4) | 12,000 |
| Supermarket 3 | (1.0,1.3) | 17,000 |
| Supermarket 3 | (1.5,2.2) | 20,000 |

The following table shows the coordinates of the candidate warehouse sites, the fixed cost of building the warehouse in millions dollar, and its respective capacity.

| <i></i> | coordinates | fixed cost | Capacity |
| --- | --- |  --- | --- |
| Warehouse 1 | (0,0) | 3 | 30,000 |
| Warehouse 2 | (0,1) | 2 | 10,000 |
| Warehouse 3 | (0,2) | 3 | 12,000 |
| Warehouse 4 | (1,0) | 1 | 8,000 |
| Warehouse 5 | (1,1) | 3 | 15,000 |
| Warehouse 6 | (1,2) | 3 | 22,000 |
| Warehouse 7 | (2,0) | 4 | 10,000 |
| Warehouse 8 | (2,1) | 3 | 30,000 |
| Warehouse 9 | (2,2) | 2 | 20,000 |

The transport cost is one million dollar per kilometer.

Formulate an ILP model and solve it.

In [4]:
from pyomo.environ import *
import math

# -----------------------------
# Data
# -----------------------------
supermarkets = {
    'S1': {'coord': (0, 1.5), 'demand': 16000},
    'S2': {'coord': (2.5, 1.2), 'demand': 18000},
    'S3': {'coord': (4, 4),    'demand': 12000},
    'S4': {'coord': (1.0, 1.3), 'demand': 17000},
    'S5': {'coord': (1.5, 2.2), 'demand': 20000}
}

warehouses = {
    'W1': {'coord': (0,0), 'fixed': 3, 'cap': 30000},
    'W2': {'coord': (0,1), 'fixed': 2, 'cap': 10000},
    'W3': {'coord': (0,2), 'fixed': 3, 'cap': 12000},
    'W4': {'coord': (1,0), 'fixed': 1, 'cap': 8000},
    'W5': {'coord': (1,1), 'fixed': 3, 'cap': 15000},
    'W6': {'coord': (1,2), 'fixed': 3, 'cap': 22000},
    'W7': {'coord': (2,0), 'fixed': 4, 'cap': 10000},
    'W8': {'coord': (2,1), 'fixed': 3, 'cap': 30000},
    'W9': {'coord': (2,2), 'fixed': 2, 'cap': 20000}
}

# Compute transport costs (1M per km)
def distance(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

cost = {(i,j): distance(supermarkets[i]['coord'], warehouses[j]['coord'])
        for i in supermarkets for j in warehouses}

# -----------------------------
# Model
# -----------------------------
model = ConcreteModel()

# Sets
model.I = Set(initialize=supermarkets.keys())
model.J = Set(initialize=warehouses.keys())

# Parameters
model.demand = Param(model.I, initialize={i: supermarkets[i]['demand'] for i in supermarkets})
model.capacity = Param(model.J, initialize={j: warehouses[j]['cap'] for j in warehouses})
model.fixed = Param(model.J, initialize={j: warehouses[j]['fixed'] for j in warehouses})
model.cost = Param(model.I, model.J, initialize=cost)

# Decision variables
model.x = Var(model.I, model.J, within=NonNegativeReals)
model.y = Var(model.J, within=Binary)

# Objective
model.obj = Objective(
    expr = sum(model.fixed[j]*model.y[j] for j in model.J) +
           sum(model.cost[i,j]*model.x[i,j] for i in model.I for j in model.J),
    sense = minimize
)

# Demand satisfaction
def demand_rule(model, i):
    return sum(model.x[i,j] for j in model.J) == model.demand[i]
model.Demand = Constraint(model.I, rule=demand_rule)

# Warehouse capacity
def capacity_rule(model, j):
    return sum(model.x[i,j] for i in model.I) <= model.capacity[j] * model.y[j]
model.Capacity = Constraint(model.J, rule=capacity_rule)

# -----------------------------
# Solve
# -----------------------------
solver = SolverFactory('glpk')  # or 'cbc'
solver.solve(model, tee=False)

# -----------------------------
# Results
# -----------------------------
print("\nWarehouses opened:")
for j in model.J:
    if model.y[j]() > 0.5:
        print(f"  {j} (capacity {model.capacity[j]})")

print("\nAllocations (supermarket -> warehouse -> quantity):")
for i in model.I:
    for j in model.J:
        if model.x[i,j]() > 0:
            print(f"  {i} -> {j}: {model.x[i,j]():,.0f} units")

print(f"\nTotal cost (millions USD): {model.obj():.3f}")



Warehouses opened:
  W2 (capacity 10000)
  W3 (capacity 12000)
  W5 (capacity 15000)
  W6 (capacity 22000)
  W8 (capacity 30000)
  W9 (capacity 20000)

Allocations (supermarket -> warehouse -> quantity):
  S1 -> W2: 10,000 units
  S1 -> W3: 6,000 units
  S2 -> W8: 18,000 units
  S3 -> W9: 12,000 units
  S4 -> W5: 15,000 units
  S4 -> W6: 2,000 units
  S5 -> W6: 12,000 units
  S5 -> W9: 8,000 units

Total cost (millions USD): 68320.752
